In [1]:
"""
Evaluate syntactic validity of the SFT-tuned PocketCoder model using instruction prompts\n(matched to the SFT training format, per the paper's protocol-matching evaluation).

This is a CUSTOM nanoGPT-style architecture (not a native `transformers` model),
so it can't be loaded with AutoModel.from_pretrained(). We rebuild the exact
GPTConfig/GPT classes used in training, download config.json + model.safetensors
from the Hugging Face repo, generate completions for a batch of prompts, and
score each generation with Python's own `ast.parse()` to see if it's valid syntax.

Run locally (not inside a network-sandboxed environment) since it needs to reach
huggingface.co:

    pip install torch transformers huggingface_hub safetensors

    export HF_TOKEN="hf_xxx"           # only needed if the repo is private
    python evaluate_syntax_validity.py
"""

import ast
import re
import json
import math
import os
import textwrap
import traceback
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

# --------------------------------------------------------------------------
# Config — edit these two lines
# --------------------------------------------------------------------------
REPO_ID = "Ananda100/100m-sft-python"  # <-- confirm this points at your SFT checkpoint, not pretrained/distilled
HF_TOKEN = os.environ.get("HF_TOKEN")  # rotate any previously-hardcoded token; set via env var instead

TOKENIZER_NAME = "deepseek-ai/deepseek-coder-6.7b-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [2]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(torch.nn.functional, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer(
                "bias",
                torch.tril(torch.ones(config.block_size, config.block_size)).view(
                    1, 1, config.block_size, config.block_size
                ),
            )

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=True,
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 32256
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(
            dict(
                wte=nn.Embedding(config.vocab_size, config.n_embd),
                wpe=nn.Embedding(config.block_size, config.n_embd),
                drop=nn.Dropout(config.dropout),
                h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                ln_f=LayerNorm(config.n_embd, config.bias),
            )
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        dev = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, (
            f"sequence length {t} exceeds block_size {self.config.block_size}"
        )
        pos = torch.arange(0, t, dtype=torch.long, device=dev)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_token_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            if eos_token_id is not None and idx.size(0) == 1 and idx_next.item() == eos_token_id:
                break
        return idx


# --------------------------------------------------------------------------
# Load model + tokenizer from the Hub
# --------------------------------------------------------------------------

def load_model_from_hub(repo_id: str, token: str | None):
    print(f"Downloading config.json and model.safetensors from {repo_id} ...")
    config_path = hf_hub_download(repo_id=repo_id, filename="config.json", token=token)
    weights_path = hf_hub_download(repo_id=repo_id, filename="model.safetensors", token=token)

    with open(config_path) as f:
        cfg_dict = json.load(f)
    config = GPTConfig(**cfg_dict)
    print("Loaded config:", asdict(config))

    model = GPT(config)
    state_dict = load_file(weights_path)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print("WARNING - missing keys:", missing)
    if unexpected:
        print("WARNING - unexpected keys:", unexpected)

    model.to(DEVICE)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model loaded: {n_params:.2f}M params on {DEVICE}")
    return model, config


def load_tokenizer(repo_id: str, token: str | None):
    # Try the tokenizer files saved in the repo itself first (matches training exactly);
    # fall back to the original DeepSeek Coder tokenizer if the repo doesn't have them.
    try:
        tok = AutoTokenizer.from_pretrained(repo_id, token=token, trust_remote_code=True)
        print(f"Loaded tokenizer from {repo_id}")
    except Exception:
        print(f"No tokenizer in {repo_id}, falling back to {TOKENIZER_NAME}")
        tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME, trust_remote_code=True)
    return tok


# --------------------------------------------------------------------------
# Generation
# --------------------------------------------------------------------------

# --------------------------------------------------------------------------
# SFT-specific: instruction prompt template + code extraction
# --------------------------------------------------------------------------
# Must match the exact format used during SFT training (### Problem / ### Solution).
SFT_PROMPT_TEMPLATE = "### Problem\n{instruction}\n### Solution\n```python\n"


def format_instruction_prompt(instruction: str) -> str:
    return SFT_PROMPT_TEMPLATE.format(instruction=instruction)


def extract_code(generated_text: str, prompt: str) -> str:
    """
    Strip the instruction scaffolding and any markdown fences so we only
    run ast.parse() on the model's actual generated code, not the prompt
    template itself (which is not valid Python on its own).

    The prompt already opens a ```python fence, so whatever the model
    generates after the prompt IS the code. If the model closes the fence,
    cut there; otherwise use the (possibly truncated) completion as-is -
    a truncated/degenerate completion correctly failing ast.parse() is
    the expected and desired behavior, not a bug to work around.
    """
    if generated_text.startswith(prompt):
        completion = generated_text[len(prompt):]
    elif "```python" in generated_text:
        completion = generated_text.split("```python", 1)[1]
    elif "```" in generated_text:
        completion = generated_text.split("```", 1)[1]
    else:
        completion = generated_text

    if "```" in completion:
        completion = completion.split("```", 1)[0]
    return completion


def generate_code(model, tokenizer, instruction, max_new_tokens=200, temperature=0.8, top_k=50):
    """
    instruction: a natural-language problem statement (e.g. "Write a function
    that returns all prime numbers less than n."). This gets wrapped in the
    SFT instruction template before being fed to the model.
    """
    prompt = format_instruction_prompt(instruction)
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        out = model.generate(
            idx,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            eos_token_id=tokenizer.eos_token_id,
        )
    full_text = tokenizer.decode(out[0].tolist(), skip_special_tokens=True)
    return full_text, prompt


PROMPTS = [
    # --- sorting / searching (18) ---
    'Write a function that sorts a list using bubble sort.',
    'Write a function that sorts a list using quicksort.',
    'Write a function that sorts a list using merge sort.',
    'Write a function that sorts a list using insertion sort.',
    'Write a function that performs binary search on a sorted list.',
    'Write a function that performs linear search on a list.',
    'Write a function that finds the k-th smallest element in an unsorted list.',
    'Write a function that merges two sorted lists into one sorted list.',
    'Write a function that checks if a list is sorted in ascending order.',
    'Write a function that sorts a list of tuples by their second element.',
    'Write a function that finds the median of a list of numbers.',
    'Write a function that implements heap sort.',
    'Write a function that returns the top k largest elements from a list.',
    'Write a function that performs a binary search and returns the insertion index if not found.',
    'Write a function that sorts a dictionary by its values.',
    'Write a function that finds the second largest number in a list.',
    'Write a function that removes duplicates from a sorted list in place.',
    'Write a function that finds the intersection of two sorted lists.',

    # --- math / number theory (18) ---
    'Write a function that returns all prime numbers less than n.',
    'Write a function that checks if a number is prime.',
    'Write a function that computes the factorial of a number.',
    'Write a function that computes the nth Fibonacci number.',
    'Write a function that computes the greatest common divisor of two numbers.',
    'Write a function that computes the least common multiple of two numbers.',
    'Write a function that checks if a number is a perfect square.',
    'Write a function that checks if a number is an Armstrong number.',
    'Write a function that computes the sum of digits of a number.',
    'Write a function that reverses the digits of an integer.',
    'Write a function that converts a decimal number to binary.',
    'Write a function that converts a binary string to a decimal integer.',
    'Write a function that computes the power of a number using recursion.',
    'Write a function that checks if two numbers are coprime.',
    'Write a function that generates the first n Fibonacci numbers as a list.',
    'Write a function that checks if a number is a palindrome.',
    'Write a function that computes the standard deviation of a list of numbers.',
    'Write a function that rounds a number to the nearest multiple of a given base.',

    # --- string manipulation (18) ---
    'Write a function that checks if a string is a palindrome, ignoring spaces and case.',
    'Write a function that reverses a string.',
    'Write a function that counts the number of words in a string.',
    'Write a function that checks if two strings are anagrams of each other.',
    'Write a function that removes all vowels from a string.',
    'Write a function that capitalizes the first letter of every word in a sentence.',
    'Write a function that finds the longest common substring between two strings.',
    'Write a function that checks if a string contains only digits.',
    'Write a function that counts the frequency of each character in a string.',
    'Write a function that removes duplicate characters from a string while preserving order.',
    'Write a function that converts a camelCase string to snake_case.',
    'Write a function that checks if a string is a valid IPv4 address.',
    'Write a function that extracts all email addresses from a block of text.',
    'Write a function that compresses a string using run-length encoding.',
    'Write a function that finds the longest palindromic substring in a string.',
    'Write a function that checks if parentheses in a string are balanced.',
    'Write a function that splits a string into chunks of a given size.',
    'Write a function that returns the most frequent word in a piece of text.',

    # --- data structures (18) ---
    'Write a class that implements a stack with push, pop, and peek methods.',
    'Write a class that implements a queue using two stacks.',
    'Write a class that implements a singly linked list with an append method.',
    'Write a class that implements a binary search tree with insert and search methods.',
    'Write a function that checks if a binary tree is balanced.',
    'Write a function that performs a breadth-first traversal of a graph.',
    'Write a function that performs a depth-first traversal of a graph.',
    'Write a class that implements a min-heap with push and pop operations.',
    'Write a function that detects a cycle in a linked list.',
    'Write a class that implements a hash map from scratch using separate chaining.',
    'Write a function that finds the lowest common ancestor of two nodes in a binary tree.',
    'Write a class that implements a trie for storing and searching words.',
    'Write a function that converts a binary tree to its mirror image.',
    'Write a function that computes the height of a binary tree.',
    'Write a class that implements a circular buffer with a fixed capacity.',
    'Write a function that checks if a binary tree is a valid binary search tree.',
    'Write a class that implements an LRU cache with get and put methods.',
    'Write a function that flattens a nested list of arbitrary depth.',

    # --- OOP / classes (18) ---
    'Write a class that represents a bank account with deposit and withdraw methods.',
    'Write a class that represents a rectangle with methods to compute area and perimeter.',
    'Write a class hierarchy where Dog and Cat inherit from an Animal base class.',
    'Write a class that implements the singleton design pattern.',
    'Write a class that represents a 2D vector and supports addition using operator overloading.',
    'Write a class that represents an employee with a method to give a raise.',
    'Write a class that implements a simple observer pattern.',
    'Write a class that represents a playing card with suit and rank attributes.',
    'Write a class that implements a factory for creating different shapes.',
    'Write a class that represents a matrix and supports matrix multiplication.',
    'Write a class that represents a polynomial and supports evaluation at a given x.',
    'Write a class that implements a simple state machine with transitions.',
    'Write a class that represents a temperature and supports conversion between Celsius and Fahrenheit.',
    'Write a class that implements a context manager for timing code execution.',
    'Write a class that represents a shopping cart with add and remove item methods.',
    'Write a class that represents a student with a method to compute their GPA.',
    'Write a class that implements an iterator over a custom range of numbers.',
    'Write a class that represents a simple to-do list with add, remove, and complete methods.',

    # --- file / I/O handling (14) ---
    'Write a function that reads a text file and counts the number of lines.',
    'Write a function that reads a CSV file and returns its contents as a list of dictionaries.',
    'Write a function that writes a list of strings to a text file, one per line.',
    'Write a function that copies the contents of one file to another.',
    'Write a function that reads a JSON file and returns its contents as a dictionary.',
    'Write a function that lists all files with a given extension in a directory.',
    'Write a function that merges the contents of multiple text files into one.',
    'Write a function that reads a file and returns the number of words it contains.',
    'Write a function that appends a line to a log file with a timestamp.',
    'Write a function that reads a large file line by line without loading it all into memory.',
    'Write a function that checks if a file exists before attempting to open it.',
    'Write a function that writes a dictionary to a JSON file.',
    'Write a function that deletes all files in a directory older than a given number of days.',
    'Write a function that reads a configuration file and returns its key-value pairs.',

    # --- error handling / validation (14) ---
    'Write a function that safely divides two numbers and handles division by zero.',
    'Write a function that validates whether an email address has a valid format.',
    'Write a function that retries a network request up to three times before failing.',
    'Write a function that validates that a given input is a positive integer.',
    'Write a custom exception class for handling invalid user input.',
    'Write a function that validates a password meets minimum security requirements.',
    'Write a function that safely converts a string to an integer, returning None on failure.',
    'Write a function that checks if a value is within a given numeric range and raises an error if not.',
    'Write a function that validates a JSON string against a required set of keys.',
    'Write a function that catches and logs any exception raised inside another function.',
    'Write a function that ensures a list is not empty before processing it.',
    'Write a function that validates a date string is in the format YYYY-MM-DD.',
    'Write a function that checks whether a given URL is well-formed.',
    'Write a function that gracefully handles a missing file and returns a default value.',

    # --- data processing / typing / ML-ish (18) ---
    'Write a function that computes the mean and standard deviation of a list of numbers.',
    'Write a function that normalizes a list of numbers to values between 0 and 1.',
    'Write a function that splits a dataset into training and test sets.',
    'Write a function that computes the accuracy of a classifier given predictions and true labels.',
    'Write a function that one-hot encodes a list of categorical labels.',
    'Write a function that computes the dot product of two vectors.',
    'Write a function that removes outliers from a list of numbers using the IQR method.',
    'Write a function that groups a list of dictionaries by a given key.',
    'Write a function that computes a moving average over a list of numbers.',
    'Write a function that tokenizes a sentence into a list of lowercase words.',
    'Write a function that builds a vocabulary dictionary from a list of sentences.',
    'Write a function that pads a list of sequences to the same length.',
    'Write a function that computes the cosine similarity between two vectors.',
    'Write a function that shuffles a dataset and its corresponding labels together.',
    'Write a function that batches a list of items into fixed-size groups.',
    'Write a function that computes precision and recall from predictions and true labels.',
    'Write a function that scales a list of numbers using min-max normalization.',
    'Write a function that computes the confusion matrix for binary classification.',

    # --- web / API / networking (14) ---
    'Write a function that fetches JSON data from a given URL.',
    'Write a function that sends a POST request with a JSON payload to an API.',
    'Write a function that checks whether a website is reachable.',
    'Write a function that downloads a file from a URL and saves it locally.',
    'Write a function that parses query parameters from a URL string.',
    'Write a function that builds a URL with query parameters from a dictionary.',
    'Write a function that checks the status code of an HTTP response and raises an error if it failed.',
    'Write a function that fetches multiple URLs concurrently and returns their responses.',
    'Write a function that implements simple rate limiting for outgoing API requests.',
    'Write a function that parses a JSON API response and extracts a specific field.',
    'Write a function that uploads a file to a remote server using an HTTP request.',
    'Write a function that implements a basic in-memory cache for API responses.',
    'Write a function that validates an API key before processing a request.',
    'Write a function that retries an HTTP request with exponential backoff.',

    # --- utility / decorators / generators (14) ---
    'Write a decorator that times how long a function takes to run.',
    'Write a decorator that caches the results of a function call.',
    'Write a decorator that retries a function a fixed number of times on failure.',
    'Write a generator function that yields Fibonacci numbers indefinitely.',
    'Write a generator function that yields batches of items from a list.',
    'Write a decorator that logs the arguments and return value of a function.',
    'Write a function that returns a memoized version of a given function.',
    'Write a generator function that reads a large file lazily line by line.',
    'Write a decorator that enforces a rate limit on how often a function can be called.',
    'Write a generator function that yields all prime numbers indefinitely.',
    "Write a decorator that validates the types of a function's arguments.",
    'Write a context manager that suppresses a specific type of exception.',
    'Write a function that returns a deep copy of a nested data structure.',
    'Write a function that measures the memory usage of a function call.',

    # --- tricky / edge-case instructions (16) ---
    'Write a function that finds all pairs in a list that sum to a target value, handling duplicates correctly.',
    'Write a function that merges overlapping intervals in a list of (start, end) tuples.',
    'Write a function that detects if a list contains a cycle when treated as a linked list of indices.',
    'Write a function that finds the longest increasing subsequence in a list of numbers.',
    'Write a function that solves the classic knapsack problem given weights, values, and a capacity.',
    'Write a function that computes the edit distance between two strings.',
    'Write a function that generates all valid combinations of balanced parentheses for a given n.',
    'Write a function that finds the shortest path between two nodes in a weighted graph.',
    'Write a function that checks if a sudoku board is valid.',
    'Write a function that solves the N-Queens problem and returns all valid solutions.',
    'Write a function that finds the maximum sum of a contiguous subarray.',
    'Write a function that determines if a string can be segmented into a sequence of dictionary words.',
    'Write a function that rotates a matrix by 90 degrees in place.',
    'Write a function that finds all permutations of a list without using itertools.',
    'Write a function that implements a basic LFU cache with get and put methods.',
    'Write a function that serializes and deserializes a binary tree to and from a string.',

    # --- concurrency / misc utility (20) ---
    'Write a function that runs two tasks concurrently using threads and waits for both to finish.',
    'Write a function that uses a lock to safely increment a shared counter from multiple threads.',
    'Write a function that processes a list of items in parallel using a process pool.',
    'Write a function that implements a simple producer-consumer pattern using a queue.',
    'Write a function that computes the union of two sets.',
    'Write a function that computes the symmetric difference between two sets.',
    'Write a function that converts a list of key-value pairs into a dictionary.',
    'Write a function that returns the n most common elements in a list.',
    'Write a function that checks if two lists have the same elements regardless of order.',
    'Write a function that computes the running total of a list of numbers.',
    'Write a function that finds the missing number in a list containing n distinct numbers from 0 to n.',
    'Write a function that generates a random password of a given length with letters, digits, and symbols.',
    'Write a function that converts seconds into a formatted string of hours, minutes, and seconds.',
    'Write a function that computes the Hamming distance between two equal-length strings.',
    'Write a function that finds all divisors of a given number.',
    'Write a function that checks if a given year is a leap year.',
    'Write a function that converts a list of numbers into their cumulative sum.',
    'Write a function that implements a basic command-line argument parser without using argparse.',
    'Write a function that computes the transpose of a 2D list without using external libraries.',
    'Write a function that deep merges two nested dictionaries.',

]


# --------------------------------------------------------------------------
# Syntax validity scoring
# --------------------------------------------------------------------------

def classify_syntax_error(code: str, error: SyntaxError) -> str:
    msg = str(error).lower()
    if "unterminated string" in msg or "eol while scanning" in msg:
        return "unterminated_string"
    if "unexpected eof" in msg or "unexpected end of file" in msg:
        return "unexpected_eof_truncation"
    if "indent" in msg:
        return "indentation_error"
    if "unmatched" in msg or "was never closed" in msg or "closing parenthesis" in msg:
        return "unbalanced_brackets"
    if "invalid syntax" in msg:
        return "invalid_syntax_other"
    return "other"


def check_syntax(code: str):
    """Returns (is_valid, error_category or None, error_message or None)."""
    try:
        ast.parse(code)
        return True, None, None
    except SyntaxError as e:
        return False, classify_syntax_error(code, e), str(e)
    except Exception as e:  # tokenize errors etc.
        return False, "other", str(e)


def evaluate(model, tokenizer, prompts=PROMPTS, max_new_tokens=200, temperature=0.8, top_k=50,
             samples_per_prompt=1, verbose=True):
    """
    samples_per_prompt > 1 lets you generate multiple completions per prompt
    (useful since sampling is stochastic) and report validity rate across all samples.
    """
    results = []
    total = len(prompts) * samples_per_prompt
    done = 0
    for instruction in prompts:
        for _ in range(samples_per_prompt):
            done += 1
            full_text, formatted_prompt = generate_code(
                model, tokenizer, instruction, max_new_tokens, temperature, top_k
            )
            code_only = extract_code(full_text, formatted_prompt)
            is_valid, category, err_msg = check_syntax(code_only)
            results.append(
                {
                    "instruction": instruction,
                    "full_generation": full_text,
                    "extracted_code": code_only,
                    "is_valid": is_valid,
                    "error_category": category,
                    "error_message": err_msg,
                }
            )

            if verbose:
                print("=" * 70)
                print(f"[{done}/{total}] INSTRUCTION:", repr(instruction))
                print("-" * 70)
                print(code_only)
                print("-" * 70)
                status = "VALID" if is_valid else f"INVALID ({category}: {err_msg})"
                print("SYNTAX:", status)

    n_valid = sum(r["is_valid"] for r in results)
    n_total = len(results)
    rate = n_valid / n_total if n_total else 0.0

    print("\n" + "=" * 70)
    print(f"OVERALL SYNTAX VALIDITY: {n_valid}/{n_total} = {rate:.1%}")
    print(f"(across {len(prompts)} distinct prompts x {samples_per_prompt} sample(s) each)")

    error_counts = {}
    for r in results:
        if not r["is_valid"]:
            error_counts[r["error_category"]] = error_counts.get(r["error_category"], 0) + 1
    if error_counts:
        print("\nError breakdown:")
        for cat, count in sorted(error_counts.items(), key=lambda x: -x[1]):
            print(f"  {cat}: {count} ({count / n_total:.1%} of all generations)")

    if samples_per_prompt > 1:
        print("\nPer-prompt validity:")
        for instruction in prompts:
            prompt_results = [r for r in results if r["instruction"] == instruction]
            v = sum(r["is_valid"] for r in prompt_results)
            print(f"  {v}/{len(prompt_results)}  {instruction[:50]!r}")

    return results, rate


if __name__ == "__main__":
    model, config = load_model_from_hub(REPO_ID, HF_TOKEN)
    tokenizer = load_tokenizer(REPO_ID, HF_TOKEN)

    # Bump samples_per_prompt to e.g. 3 to average out sampling noise per prompt
    results, rate = evaluate(model, tokenizer, samples_per_prompt=1)

    with open("sft_syntax_validity_results.json", "w") as f:
        json.dump({"validity_rate": rate, "results": results}, f, indent=2)
    print("\nSaved detailed results to sft_syntax_validity_results.json")

config.json:   0%|          | 0.00/130 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  482MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded config: {'block_size': 512, 'vocab_size': 32022, 'n_layer': 10, 'n_head': 12, 'n_embd': 768, 'dropout': 0.1, 'bias': True}
Model loaded: 95.87M params on cuda


tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.29M [00:00<?, ?B/s]

Loaded tokenizer from Ananda100/100m-sftd-python
[1/49] PROMPT: 'import asyncio\n\nasync def'
----------------------------------------------------------------------
import asyncio

async def async_process_events(events, event_id):
    for event in events:
        await asyncio.sleep(event.data[event_id])

# Example usage:
async def test_event_with_timeout(loop):
    with asyncio.ensure_future(loop) as s:
        async with event_with_timeout(5, timeout=3) as msg:
            await msg.set_timeout(3)
    assert await asyncio.wait_for(lambda: len(s.get_events()) == 3)

async def test_event_with_timeout_concurrently(loop):
    with asyncio.ensure_future(loop) as s:
        async with event_with_timeout(5, timeout=3) as msg:
            await s.set_timeout(3)
    assert await asyncio.wait_for(lambda: len
----------------------------------------------------------------------
SYNTAX: INVALID (unbalanced_brackets: '(' was never closed (<unknown>, line 18))
[2/49] PROMPT: 'import pandas as pd\